# iNaturalist *Acacia* observations — Coimbra district

**Purpose.** Look at the citizen-science data on their own (part 1),
then clean them (part 2).

**Data.** Every *Acacia* observation in the iNaturalist place
"Coimbra, PT" (the Coimbra **district**), as downloaded
(`data/raw/citizen_science/`, see `data/README.md`).

**Part 1 — exploration (nothing changed):** 1. what comes with each
observation; 2. when; 3. several observations in the same place;
4. where; 5. quality; 6. observers.

**Part 2 — cleaning:** 7. keep dated observations; 8. remove obscured
coordinates; 9. anonymise the main observer; 10. keep only
*A. dealbata*; 11. keep only research grade; 12. positional accuracy;
13. one observation per pixel; 14. burned-area flag; 15. save.

Run this notebook before `other_plants_observations.ipynb` and the
expert notebook, which read its output.

## 0. Download and setup

**Download.** Asks the iNaturalist API for every *Acacia* observation
(genus *Acacia*, taxon 47452, all descendants) in the Coimbra district
(place 8422), all years and all quality grades, and saves it as
`data/raw/citizen_science/inaturalist_acacia_coimbra_pilot.csv`.
- Pages of 200, in order of id, waiting 1 s between requests
  (iNaturalist's rate limit).
- **Only runs if the file does not exist.** The file used in the
  thesis was downloaded on 2026-09-22 (7 943 observations); a new
  download today would return more observations, so the existing file
  is never replaced.

**Setup.** Loads the CSV and turns each observation into a point in
EPSG:3763 (PT-TM06, in metres), so distances and grid cells are in
metres.

In [ ]:
import time
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW_FILE = ROOT / "data" / "raw" / "citizen_science" / "inaturalist_acacia_coimbra_pilot.csv"

def download_acacia(path):
    params = dict(place_id=8422, taxon_id=47452, order_by="id", order="asc", per_page=200, id_above=0)
    rows = []
    while True:
        r = requests.get("https://api.inaturalist.org/v1/observations", params=params, timeout=60)
        r.raise_for_status()
        res = r.json()["results"]
        if not res:
            break
        for o in res:
            if not o.get("geojson"):
                continue
            lon, lat = o["geojson"]["coordinates"]
            t = o.get("taxon") or {}
            rows.append(dict(id=o["id"], observed_on=o.get("observed_on"), time_observed_at=o.get("time_observed_at"),
                             quality_grade=o.get("quality_grade"), taxon_id=t.get("id"), taxon_name=t.get("name"),
                             taxon_rank=t.get("rank"), common_name=t.get("preferred_common_name"),
                             latitude=lat, longitude=lon, positional_accuracy=o.get("positional_accuracy"),
                             coordinates_obscured=o.get("obscured"), place_guess=o.get("place_guess"),
                             user_login=(o.get("user") or {}).get("login"), license_code=o.get("license_code"),
                             uri=o.get("uri")))
        params["id_above"] = res[-1]["id"]
        time.sleep(1)  # iNaturalist asks for <= 60 requests/min
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(path, index=False)

if not RAW_FILE.exists():
    download_acacia(RAW_FILE)

In [ ]:
inat = pd.read_csv(RAW_FILE)
inat["observed_on"] = pd.to_datetime(inat["observed_on"], errors="coerce")
inat = gpd.GeoDataFrame(inat, geometry=gpd.points_from_xy(inat.longitude, inat.latitude), crs=4326).to_crs(3763)
print(f"{len(inat):,} observations")

## 1. What comes with each observation

One row per observation. The columns that matter later:
- `observed_on`: the date;
- `taxon_name`, `taxon_rank`: the species (or only the genus);
- `latitude`, `longitude`, `positional_accuracy` (metres, set by the
  observer), `coordinates_obscured` (location deliberately moved);
- `quality_grade`: `research` (identity confirmed by the community),
  `needs_id` or `casual`;
- `user_login`: the observer.

There is **no absence, abundance or cover**: each row only says
"*Acacia* was seen here".

In [ ]:
(inat.drop(columns="geometry").isna().mean() * 100).round(1).to_frame("pct_missing")

## 2. When

Observations per year and per month. The month matters because
*A. dealbata* flowers in winter (yellow, easy to spot): people may
record it mostly then.

In [ ]:
year = inat["observed_on"].dt.year
print("from", inat["observed_on"].min().date(), "to", inat["observed_on"].max().date())
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
year.value_counts().sort_index().loc[2010:].plot.bar(ax=axes[0], color="tab:green", title="Observations per year (from 2010)")
inat["observed_on"].dt.month.value_counts().sort_index().plot.bar(ax=axes[1], color="tab:green", title="Observations per month")
plt.tight_layout()

## 3. Several observations in the same place

A 10 m cell is one satellite pixel: observations sharing one become
the same training example. Shown for 10 m, 100 m and 1 km cells.

In [ ]:
def cell(gdf, size):
    return (gdf.geometry.x // size).astype(int).astype(str) + "_" + (gdf.geometry.y // size).astype(int).astype(str)

pd.DataFrame([dict(cell=f"{s} m", cells_with_obs=cell(inat, s).nunique(),
                   pct_obs_sharing_a_cell=round(100 * cell(inat, s).duplicated(keep=False).mean(), 1))
              for s in [10, 100, 1000]])

## 4. Where: dense areas and empty areas

Observations per 1 km cell (log colour scale; empty cells blank). An
empty cell means nobody recorded *Acacia* there, **not** that there
is none.

In [ ]:
counts = cell(inat, 1000).value_counts()
for k in [1, 10, 50]:
    print(f"Top {k:>2} cells (of {len(counts)}) hold {counts.iloc[:k].sum() / len(inat):.1%} of observations")

x, y = inat.geometry.x // 1000, inat.geometry.y // 1000
fig, ax = plt.subplots(figsize=(8, 7))
h = ax.hist2d(x, y, bins=[np.arange(x.min(), x.max() + 2), np.arange(y.min(), y.max() + 2)],
              cmin=1, norm=plt.matplotlib.colors.LogNorm(), cmap="viridis")
fig.colorbar(h[3], ax=ax, label="observations per km²")
ax.set_title("Observations per 1 km cell"); ax.set_aspect("equal"); ax.set_axis_off()

## 5. Quality

- `quality_grade`: how many were confirmed by the community.
- `coordinates_obscured`: locations moved by up to several km
  (useless for 10 m pixels).
- `positional_accuracy`: how precise the location is, when the
  observer gave it.
- Species: different *Acacia* species may look different from space.

In [ ]:
print(inat["quality_grade"].value_counts().to_dict())
print("coordinates obscured:", int(inat["coordinates_obscured"].sum()))
acc = inat["positional_accuracy"]
print(f"positional_accuracy missing: {acc.isna().mean():.1%}")
pd.cut(acc.dropna(), [0, 10, 30, 100, 1000, np.inf], include_lowest=True,
       labels=["<=10 m", "10-30 m", "30-100 m", "100 m-1 km", ">1 km"]).value_counts().sort_index()

In [ ]:
inat["taxon_name"].value_counts().to_frame("n")

## 6. Observers

One very active observer imprints their own routes on the data: the
map can show where they walked rather than where *Acacia* is (RQ3).

In [ ]:
per_user = inat["user_login"].value_counts()
print(f"{len(per_user)} observers; the top one made {per_user.iloc[0] / len(inat):.1%} of the observations")
inat["observer"] = np.where(inat["user_login"] == per_user.index[0], "top observer", "others")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
pd.crosstab(year, inat["observer"]).loc[2015:].plot.bar(stacked=True, ax=axes[0], title="Observations per year")
for grp, colour in [("others", "tab:blue"), ("top observer", "tab:orange")]:
    inat[inat["observer"] == grp].plot(ax=axes[1], markersize=2, alpha=0.5, color=colour, label=grp)
axes[1].legend(markerscale=5); axes[1].set_title("Where"); axes[1].set_axis_off()
plt.tight_layout()

# Part 2 — Cleaning (preprocessing)

From here the data are changed, one step at a time, and saved to
`data/interim/citizen_science/inat_dealbata_clean.csv`. The raw file is
never modified. Each step says what it does, why, and what changed.

Not done yet (open decisions): merge observations in the
same 10 m pixel.

## 7. Keep dated observations

**What:** read the raw file again, turn each observation into a point
in EPSG:3763 and keep only those with a date.
**Why:** the burned-area rule (step 14) compares the observation date
with fire dates.

In [ ]:
import sys

import requests
from shapely.geometry import shape

sys.path.insert(0, str(ROOT))
from src.burned_areas import burned_between, load_burned_areas
from src.grid import pixel_id

REF_YEAR = 2023  # all observations are treated as this year

raw = pd.read_csv(RAW_FILE)
acacia = raw.copy()
acacia["date"] = pd.to_datetime(acacia["observed_on"], errors="coerce")
acacia["year"] = acacia["date"].dt.year
acacia = acacia[acacia["year"].notna()]
acacia = gpd.GeoDataFrame(acacia, geometry=gpd.points_from_xy(acacia.longitude, acacia.latitude), crs=4326).to_crs(3763)
print(f"{len(raw)} → {len(acacia)} (dropped {len(raw) - len(acacia)} without a date)")

**Result.** Only 2 observations have no date.

## 8. Remove obscured coordinates

**What:** drop observations with `coordinates_obscured = True`.
**Why:** when an observer or a species has geoprivacy set, iNaturalist
moves the point to a random spot inside a ~0.2° cell (several km), so
the location is useless for 10 m pixels.

In [ ]:
before = len(acacia)
acacia = acacia[~acacia["coordinates_obscured"]]
print(f"{before} → {len(acacia)} (dropped {before - len(acacia)} obscured)")

**Result.** 43 obscured observations dropped; 7 898 remain.

## 9. Anonymise the main observer

**What:** a column `observer` with "Observer A" for the person with
the most observations and "Others" for everyone else. The username is
not kept in the cleaned file.
**Why:** the RQ3 experiments need to tell that person apart, but the
thesis must not name them.

In [ ]:
dominant = acacia["user_login"].value_counts().index[0]
acacia["observer"] = np.where(acacia["user_login"] == dominant, "Observer A", "Others")
acacia["observer"].value_counts()

**Result.** Observer A made 6 591 of the 7 898 observations (83%).

## 10. Keep only *Acacia dealbata*

**What:** keep observations identified as *A. dealbata*, at species
or subspecies level (`taxon_name` starting with "Acacia dealbata").
Other species (*A. melanoxylon*, *A. longifolia*, ...) and records
identified only to genus or subgenus are dropped.
**Why:** decided 2026-10-20 (option A, see `thesis/notes.md`): the
expert data only have *A. dealbata*, the proposal asks for
species-level mapping, and the species look different from space.
The other *Acacia* are not used as "other plants" either: the
other-plants download already excludes the whole genus.

In [ ]:
before = len(acacia)
print(acacia["taxon_name"].value_counts().to_string())
acacia = acacia[acacia["taxon_name"].str.startswith("Acacia dealbata")]
print(f"\n{before} → {len(acacia)} (dropped {before - len(acacia)} other Acacia or genus-level)")
print(acacia["observer"].value_counts().to_dict())

**Result.** 5 242 *A. dealbata* observations remain (5 240 at species
level and 2 at subspecies level); 2 656 are dropped: 2 621 other
species and 35 identified only to genus or subgenus. Observer A made
4 728 of them (90%), a larger share than among all *Acacia* (83%).

## 11. Keep only research grade

**What:** keep observations with `quality_grade = research`; drop
`casual` and `needs_id`.
**Why (decided 2026-10-29, see `thesis/notes.md`):**
- `research` means the identification was confirmed by the community
  (at least two people agree), with a photo, date, location, and a
  wild organism. It is what goes to GBIF, and what Gillespie et al.
  (2024) and Brown et al. (2025) used.
- Our `casual` *A. dealbata* are almost all observations **without a
  photo** (1 670 of 1 679 in the district, per the iNaturalist API on
  2026-10-29; 8 marked cultivated), so nobody can check the
  identification. 98% are Observer A's, mostly from 2021.
- `needs_id` have a photo but no confirmed identification yet.

**Future work:** a sensitivity check, training with and without the
`casual` and `needs_id` observations, to see whether the results
change. They can be recovered from the raw file.

In [ ]:
print(pd.crosstab(acacia["quality_grade"], acacia["observer"], margins=True))
before = len(acacia)
acacia = acacia[acacia["quality_grade"] == "research"]
print(f"\n{before} → {len(acacia)} (dropped {before - len(acacia)} casual or needs_id)")
print(f"Observer A: {(acacia.observer == 'Observer A').mean():.1%}")

**Result.** 3 441 research-grade observations remain; 1 801 are
dropped (1 678 `casual`, 123 `needs_id`). 1 648 of the 1 678 casual
ones were Observer A's, so Observer A's share drops from 90.2% to
86.9%.

## 12. Positional accuracy

**What:** `positional_accuracy` is the radius (m) of the circle in
which the plant is, as recorded by the device or the observer. A
pixel is 10 m, so a point with a large radius may fall in the wrong
pixel (an impure label; Ball et al., 2026).

**Problem:** 80% of the observations have no value, and almost all of
those are Observer A's (90% of their observations, vs. 11% for the
others). Missing does not mean badly located: it is how this person
records. Dropping them would remove Observer A.

**Test:** are the observations without a value well located? For
each accuracy group, the share of presences inside a COS 2023
"Florestas de acácias" polygon (class 5.1.1.7), compared with the
share of the district covered by that class (what random points would
give). COS only maps stands of at least 1 ha, so this is an indirect
test: it compares groups, it does not check each point.

In [ ]:
place = requests.get("https://api.inaturalist.org/v1/places/8422", timeout=60).json()["results"][0]
district = gpd.GeoSeries([shape(place["geometry_geojson"])], crs=4326).to_crs(3763).iloc[0]
COS = ROOT / "data" / "interim" / "spatial" / "COS2023v1-S2.gpkg"
cos_acacia = gpd.read_file(COS, layer="COS2023v1", where="COS23_n4_C = '5.1.1.7'",
                           bbox=district.bounds, columns=["COS23_n4_C"])
baseline = cos_acacia.clip(district).area.sum() / district.area

acacia["accuracy_group"] = (pd.cut(acacia["positional_accuracy"], [0, 10, 30, 100, np.inf], include_lowest=True,
                                   labels=["<=10 m", "10-30 m", "30-100 m", ">100 m"])
                            .cat.add_categories("missing").fillna("missing"))
acacia["in_cos_acacia"] = acacia.index.isin(gpd.sjoin(acacia[["geometry"]], cos_acacia[["geometry"]], predicate="within").index)
pct = lambda s: round(100 * s.mean(), 1)
print(f"share of the district in COS acacia (random-point baseline): {100 * baseline:.1f}%")
pd.concat([acacia.groupby("accuracy_group", observed=True).agg(n=("id", "size"), pct_in_cos_acacia=("in_cos_acacia", pct)),
           acacia[acacia.observer == "Others"].groupby("accuracy_group", observed=True)
               .agg(n_others=("id", "size"), pct_in_cos_acacia_others=("in_cos_acacia", pct))], axis=1)

**Result.**
- **Observations without a value are well located:** 18.6% fall in COS
  acacia stands, almost as many as the most precise ones (≤ 10 m,
  20.7%), and far above random points (2.3%, the share of the district
  covered by COS acacia). They are kept, marked with
  `accuracy_missing`.
- **Up to 30 m the location is good; beyond, it gets worse.** For the
  observers other than Observer A, ≤ 10 m and 10–30 m give the same
  (10.5% and 10.0%); 30–100 m and > 100 m both drop to about 6.5%.
  So 30–100 m behave like > 100 m.
- Caveats: the groups with a value are small (74–429), and the order
  is not perfect (> 100 m, 10.8%, sits above 30–100 m, 6.6%).

**Decision (2026-10-29, see `thesis/notes.md`):** drop observations
with a known accuracy above 30 m (3 pixels); keep those without a
value. The same rule applies to other plants.

In [ ]:
before = len(acacia)
acacia["accuracy_missing"] = acacia["positional_accuracy"].isna()
acacia = acacia[acacia["accuracy_missing"] | (acacia["positional_accuracy"] <= 30)]
print(f"{before} → {len(acacia)} (dropped {before - len(acacia)} with accuracy > 30 m); "
      f"without a value (kept): {acacia.accuracy_missing.sum()}")
print(f"Observer A: {(acacia.observer == 'Observer A').mean():.1%}")

**Result.** 165 observations dropped (accuracy > 30 m); 3 276
remain, of which 2 743 have no accuracy value (kept). Observer A's
share goes back up to 89.1%, since the dropped ones were mostly other
observers' (they record accuracy).

## 13. One observation per pixel

**What:** the model sees 10 m pixels, not points: two observations in
the same pixel are the same training example. Each observation gets a
`pixel` id (10 m grid in UTM 29N, EPSG:32629, the CRS of the
Sentinel-2 and TESSERA tiles; code in `src/grid.py`) and only one per
pixel is kept: the one closest in date to 2023 (the reference year;
ties: the most recent). `n_obs_in_pixel` keeps how many there were.
**Why (decided 2026-10-29, see `thesis/notes.md`):** otherwise the
most visited places count several times. Almost all repeats are
Observer A's, so keeping them would give that person's places more
weight.

In [ ]:
acacia["pixel"] = pixel_id(acacia)
acacia["n_obs_in_pixel"] = acacia.groupby("pixel")["id"].transform("size")
shared = acacia[acacia.n_obs_in_pixel > 1]
print(f"{len(shared)} observations share a pixel ({len(shared) / len(acacia):.1%}); "
      f"Observer A: {(shared.observer == 'Observer A').mean():.1%}; max per pixel: {acacia.n_obs_in_pixel.max()}")

before = len(acacia)
acacia["dist_to_ref"] = (acacia["year"] - REF_YEAR).abs()
acacia = (acacia.sort_values(["dist_to_ref", "date"], ascending=[True, False])
                .drop_duplicates("pixel").drop(columns="dist_to_ref").sort_index())
print(f"{before} → {len(acacia)} (one per pixel)")
print(f"Observer A: {(acacia.observer == 'Observer A').mean():.1%}")

**Result.** 511 observations (15.6%) shared a pixel with another,
at most 4 per pixel, and 98.2% of them were Observer A's (mostly the
same year: repeated records of the same visit). 3 000 remain, one per
pixel. Observer A's share drops slightly, to 88.2%.

## 14. Burned-area flag

**What:** a column `burned` that is `True` if the observation lies in
an area that burned **between the observation and 2023**:
- observation before 2023 → fire after the observation and up to the
  end of 2023;
- observation after 2023 → fire from the start of 2023 up to the
  observation.

Fires with a start date are compared by date; older fires only have a
year, and a fire in the same year as the observation counts as burned
(order unknown, also in `burned_same_year`).
**Why:** all observations are treated as 2023. If the place burned in
between, the 2023 pixel shows burned ground or regrowth, not what was
observed. *Acacia* is not gone after a fire (Riveiro et al., 2025);
the problem is that the pixel no longer matches the label.
**Nothing is removed:** the column lets each experiment decide.

The burned areas are ICNF's "Áreas ardidas" 1975–2025, downloaded once
to `data/external/auxiliary/icnf_burned_areas/` (see
`data/README.md`); the code is in `src/burned_areas.py`, shared with
the other-plants notebook.

In [ ]:
fires = load_burned_areas(ROOT / "data" / "external" / "auxiliary" / "icnf_burned_areas")
fires = fires[fires.intersects(district)]
print(f"burned areas intersecting the district, 1975–2025: {len(fires)}")

acacia["burned"], acacia["burned_same_year"] = burned_between(acacia, fires, REF_YEAR)
print(f"burned between observation and {REF_YEAR}: {acacia.burned.sum()} ({acacia.burned.mean():.1%}); "
      f"of which same year without a fire date: {acacia.burned_same_year.sum()}")

**Result.** Only 6 observations (0.2%) lie in an area that burned
between the observation and 2023, and no case is ambiguous. The flag
barely matters for *A. dealbata*.

**Areas burned in 2025.** The rule keeps these (the 2023 image
predates the fire), but the number is the reason the reference year is
2023 and not a later one: with 2025 images these places would be
burned ground.

In [ ]:
in_2025 = gpd.sjoin(acacia[["id", "geometry"]], fires.loc[fires["Ano"] == 2025, ["geometry"]], predicate="within")["id"].nunique()
print(f"Acacia in areas burned in 2025: {in_2025} of {len(acacia)} ({in_2025 / len(acacia):.1%})")

fig, ax = plt.subplots(figsize=(9, 9))
fires.plot(ax=ax, column="Ano", legend=True, alpha=0.5, cmap="YlOrRd", legend_kwds=dict(label="Fire year", shrink=0.6))
acacia[~acacia.burned].plot(ax=ax, markersize=2, color="tab:blue", alpha=0.6)
acacia[acacia.burned].plot(ax=ax, markersize=30, color="black", marker="x")
gpd.GeoSeries([district], crs=3763).boundary.plot(ax=ax, color="black", lw=0.8)
ax.set_title(f"Acacia and burned areas 1975–2025 (black ×: burned before 2023, {acacia.burned.sum()})"); ax.set_axis_off()

**Result.** 728 presences (24.3%) lie in areas burned in 2025 (the
2025 fires in the district). They are kept, since the 2023 image
predates the fire, but with 2025 images they would be burned ground.

## 15. Save

Only the columns needed later (no username).

In [ ]:
OUT = ROOT / "data" / "interim" / "citizen_science" / "inat_dealbata_clean.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)
keep = ["id", "observed_on", "year", "taxon_id", "taxon_name", "taxon_rank", "latitude", "longitude",
        "positional_accuracy", "quality_grade", "accuracy_missing", "observer", "pixel", "n_obs_in_pixel", "burned", "burned_same_year"]
acacia[keep].to_csv(OUT, index=False)
print(OUT.relative_to(ROOT), len(acacia), "observations")

## Check: recency vs. observer (RQ2 filter design)

**What:** the RQ2 filter (`docs/research_questions.md#rq2--data-efficiency`)
removes the oldest observations first, on the assumption that recency
is what matters for data efficiency. The design doc already states a
limitation: each level changes quantity *and* composition (age,
observers, places) at once. Observer A alone is 88.2% of these 3 000
presences and 98% of the 2021 peak (the RQ3 bias case) — if "more
recent" and "Observer A" were nearly the same thing in this dataset,
RQ2's data-efficiency result would be entangled with RQ3's
observer-bias question, and that would need to be stated explicitly
before interpreting either.
**Why now:** levels are not fixed yet ("fix levels after checking the
real counts"); this checks the confound before doing so, not a
decision in itself.

In [ ]:
overall_share = (acacia.observer == "Observer A").mean()
print(f"Observer A overall: {overall_share:.1%}")

print("\n-- oldest removed first (as worded in RQ2) --")
for cutoff in [2019, 2021, 2023]:
    kept, removed = acacia[acacia.year >= cutoff], acacia[acacia.year < cutoff]
    print(f"keep >= {cutoff}: {len(kept)} ({len(kept) / len(acacia):.1%}) | "
          f"Observer A in kept: {(kept.observer == 'Observer A').mean():.1%}, "
          f"in removed: {(removed.observer == 'Observer A').mean() if len(removed) else float('nan'):.1%}")

print("\n-- distance to 2023 (the open alternative reading of \"recent\") --")
by_dist = acacia.assign(dist=(acacia.year - REF_YEAR).abs()).sort_values("dist")
for frac in [0.855, 0.366]:
    n = round(len(acacia) * frac)
    kept, removed = by_dist.iloc[:n], by_dist.iloc[n:]
    print(f"keep closest {frac:.1%} (n={n}): "
          f"Observer A in kept: {(kept.observer == 'Observer A').mean():.1%}, "
          f"in removed: {(removed.observer == 'Observer A').mean():.1%}")

**Result.** The two axes are correlated, but not in the single
direction assumed above, and the two candidate readings of "recent"
disagree with each other. Oldest-removed-first: Observer A's share in
the kept set stays close to the 88.2% overall at the coarser cuts
(87–89% at ≥2021, ≥2019), then *drops* to 73.7% at the most aggressive
cut (keep ≥2023, 36.6% kept) — the oldest removed observations
(pre-2021) are themselves 93–97% Observer A's, since that person has
records going further back than most others. Distance-to-2023 gives a
much flatter picture at the same retention fractions (88.4% kept vs.
87.1% removed at 85.5%; 89.4% vs. 87.5% at 36.6%) — nowhere near the
73.7% seen with the other reading at the same 36.6% level. So fixing
*which* "recent" RQ2 uses (already open in the design doc) is not just
a wording choice: it changes how much the recency filter is also,
incidentally, an observer filter. Whichever reading and levels are
chosen, Observer A's share at each kept level should be reported
alongside the RQ2 result. This is a check of mine, not grounded in
literature; no source was found that examines this specific
interaction.

## Summary

**Part 1 — exploration.**

- **7 943 observations**, 1992–2026, from 309 observers. Presence
  only: no absences.
- **When:** very uneven. 2021 alone holds 42.9%; 2024 15.4%; 2023
  only 3.5%; almost nothing before 2019. By month, March is the peak
  (16.4%, *A. dealbata* flowering), but every month has records.
- **One observer made 83.2%** of the observations, and almost all of
  the 2021 peak.
- **Same place:** 37.2% share a 10 m pixel with another observation.
- **Clustered:** 50 of 756 occupied 1 km cells hold 66.2% of the
  observations.
- **Quality:** 5 236 `research`, 2 449 `casual`, 258 `needs_id`. 43
  have obscured coordinates. `positional_accuracy` is missing for
  79.8%; where given, most are within 10 m.
- **Species:** *A. dealbata* 5 254, *A. melanoxylon* 1 898,
  *A. longifolia* 517, the rest few; 36 only at genus or section
  level.

**Part 2 — cleaning.** 2 observations without a date and 43 with
obscured coordinates dropped; main observer anonymised as Observer A;
only *A. dealbata* kept (7 898 → 5 242); only research grade kept
(5 242 → 3 441; the casual ones are almost all without a photo);
positional accuracy above 30 m dropped (3 441 → 3 276), observations
without a value kept and marked (`accuracy_missing`), after a test
showing they fall in COS acacia stands almost as often as the most
precise ones; one observation per 10 m pixel (3 276 → 3 000);
`burned` flag added (6 observations, 0.2%; nothing removed). 728
(24.3%) lie in areas burned in 2025, kept. Observer A made 88.2%.
Result: 3 000 observations in
`data/interim/citizen_science/inat_dealbata_clean.csv`. Future work:
sensitivity check with and without `casual` and `needs_id`.